In [46]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import ToTensor, ToPILImage
from PIL import Image
from utils.datasets import load_data, HandwrittenData
from utils.preprocessing import process_img
from core.tokenizer import Tokenizer
from core.crnn import HandwrittenCRNN
from torch.utils.data import RandomSampler
from core.run_inference import CRNNInferencer

In [47]:
data_dict_parents = "dataset_kaggle"
data_dict_file = "test_split.json"
data_dict_path = os.path.join(data_dict_parents, data_dict_file)

datadict = load_data(data_dict_path)
kaggle_path = "/kaggle/input/datasets/verack/spanish-handwritten-characterswords/datos_entrenamiento_augmented/PERFECT_CUT_a_z_1_9_aug_SYNTHETIC/"
datadict.data = {v.replace(kaggle_path,""):k for v, k in datadict.data.items()} 
datadict.index = [v.replace(kaggle_path,"") for v in datadict.index] 

In [48]:
tokens_file = os.path.join("include", "tokens.json")
tokenizer = Tokenizer(src=tokens_file)

In [49]:
data_parents = os.path.join("dataset_kaggle", "images")
dataset = HandwrittenData(
    datadict=datadict, data_path=data_parents, transform=process_img
)

In [50]:
model_path = os.path.join("include", "crnn_best_weights.pth")
inference = CRNNInferencer(model_path=model_path, tokenizer=tokenizer, num_classes=tokenizer.ntokens, device="cpu")

Inference in device: cpu


In [102]:
import gradio as gr
from PIL import Image
import tqdm as notebook_tqdm

to_tensor = ToTensor()
def predict_from_notebook(canvas_img) -> str:

    if isinstance(canvas_img, dict):
        img = canvas_img['composite']
    else:
        img = canvas_img

    if img == None:
        return "Draw something first..."
    
    img = process_img(img)
    img_tensor = to_tensor(img)
    prediction = inference.predict(img_tensor)
    
    return prediction

In [117]:
with gr.Blocks(theme=gr.themes.Soft()) as iface:
    
    # Encabezado principal
    gr.Markdown("## ✍️ Reconocimiento de Caligrafía (OCR)")
    gr.Markdown("Dibuja una palabra en el lienzo de abajo y presiona **Predecir** para que el modelo la lea.")
    
    with gr.Accordion("📌 Consejos para una mejor predicción", open=False):
        gr.Markdown("""
        * **Aprovecha el espacio:** Intenta que la palabra ocupe bien el centro del lienzo.
        * **Letra clara:** Evita tachones. Si te equivocas, usa el botón de *Limpiar*.
        * **Grosor del trazo:** Usa trazos gruesos para un mejor reconocimiento.
        """)
    
    #
    dibujo = gr.Sketchpad(
        image_mode='L', 
        brush=gr.Brush(colors=["black"]), 
        type="pil", 
        label="Lienzo de dibujo",
        height=300, 
        canvas_size=(1000, 300) 
    )
    
    with gr.Row():
        btn_limpiar = gr.Button("🗑️ Limpiar Lienzo", variant="secondary")
        btn_predecir = gr.Button("✨ Predecir Palabra", variant="primary")
        
    salida = gr.Textbox(
        label="Texto Predicho", 
        placeholder="El resultado aparecerá aquí...", 
    )
        
    # Pred
    btn_predecir.click(
        fn=predict_from_notebook, 
        inputs=dibujo, 
        outputs=salida
    )
    
    # Clean
    btn_limpiar.click(
        fn=lambda: (None, ""), 
        inputs=None, 
        outputs=[dibujo, salida]
    )

# Lanzamos la app
iface.launch(inline=True)

/tmp/ipykernel_101987/4233907076.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as iface:


* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.
